In [27]:
%%writefile test_hyp.py

from hypothesis import given
from hypothesis import strategies as st

NUM_LIMIT = 2**53

@given(st.lists(st.integers()))
def test_reverse_twice_is_original(xs):
    assert list(reversed(list(reversed(xs)))) == xs

def average(numbers):
    if not numbers:
        raise ValueError("Cannot calculate the average of an empty list.")
    if abs(sum(numbers)) > NUM_LIMIT or abs(len(numbers)) > NUM_LIMIT:
        raise ValueError("Precision Lost")
    return sum(numbers) / len(numbers)

rand_vals = st.lists(st.integers(min_value=1, max_value=1000), min_size=1, max_size=NUM_LIMIT)

@given(rand_vals)
def test_average(numbers):
    result = average(numbers)
    assert min(numbers) <= result <= max(numbers)

Overwriting test_hyp.py


In [2]:
%%writefile test_state.py

from hypothesis.stateful import RuleBasedStateMachine, rule, invariant
from hypothesis import strategies as st

class DataStore:
    def __init__(self):
        self.records = {}

    def insert(self, key, value):
        self.records[key] = value

    def delete(self, key):
        self.records.pop(key, None)

    def update(self, key, value):
        if key in self.records:
            self.records[key] = value

class DataStoreMachine(RuleBasedStateMachine):
    def __init__(self):
        super().__init__()
        self.store = DataStore()
        self.model = {}

    @rule(key=st.text(min_size=1, max_size=3), value=st.integers())
    def insert(self, key, value):
        self.store.insert(key, value)
        self.model[key] = value

    @rule(key=st.text(min_size=1, max_size=3))
    def delete(self, key):
        self.store.delete(key)
        self.model.pop(key, None)

    @rule(key=st.text(min_size=1, max_size=3), value=st.integers())
    def update(self, key, value):
        self.store.update(key, value)
        if key in self.model:
            self.model[key] = value

    @invariant()
    def store_matches_model(self):
        assert self.store.records == self.model


TestDataStore = DataStoreMachine.TestCase

Overwriting test_state.py
